In [1]:
import pandas as pd
import numpy as np

C:\Users\swati\AppData\Roaming\Python\Python311\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\swati\AppData\Roaming\Python\Python311\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [2]:
temp_df = pd.read_csv("IMDB Dataset.csv")

In [3]:
df = temp_df.iloc[:1000]

In [4]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
df.drop_duplicates(inplace=True)

In [6]:
import re
def remove_tags(text):
    text = re.sub(r'<.*?>', '', text)
    return text

In [7]:
df['review'] = df['review'].apply(remove_tags)

In [8]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. The filming tec...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
995,Nothing is sacred. Just ask Ernie Fosselius. T...,positive
996,I hated it. I hate self-aware pretentious inan...,negative
997,I usually try to be professional and construct...,negative
998,If you like me is going to see this in a film ...,negative


In [9]:
df['review'] = df['review'].apply(lambda x:x.lower())


In [10]:
from nltk.corpus import stopwords

sw_list = stopwords.words('english')

df['review'] = df['review'].apply(lambda x: [item for item in x.split() if item not in sw_list]).apply(lambda x:" ".join(x))

In [11]:
df['review']

0      one reviewers mentioned watching 1 oz episode ...
1      wonderful little production. filming technique...
2      thought wonderful way spend time hot summer we...
3      basically there's family little boy (jake) thi...
4      petter mattei's "love time money" visually stu...
                             ...                        
995    nothing sacred. ask ernie fosselius. days, eve...
996    hated it. hate self-aware pretentious inanity ...
997    usually try professional constructive criticiz...
998    like going see film history class something li...
999    like zoology textbook, given depiction animals...
Name: review, Length: 1000, dtype: str

In [12]:
import gensim

c:\ProgramData\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated and will be removed in a future release
  "class": algorithms.Blowfish,


In [13]:
from nltk import sent_tokenize
from gensim.utils import simple_preprocess

In [14]:
story = []
for doc in df['review']:
    raw_sent = sent_tokenize(doc)
    for sent in raw_sent:
        story.append(simple_preprocess(sent))


In [15]:
model = gensim.models.Word2Vec(
    window=10,
    min_count=2
)


In [16]:
model.build_vocab(story)


In [17]:
model.train(story, total_examples=model.corpus_count, epochs=model.epochs)


(554594, 621105)

In [18]:
len(model.wv.index_to_key)


9395

In [19]:
def document_vector(doc):
    # remove out-of-vocabulary words
    doc = [word for word in doc.split() if word in model.wv.index_to_key]
    return np.mean(model.wv[doc], axis=0)

In [20]:
document_vector(df['review'].values[0])


array([-0.2584384 ,  0.33628497,  0.15411253,  0.08223379,  0.03106987,
       -0.5955327 ,  0.21025276,  0.77334595, -0.24835569, -0.2774411 ,
       -0.2826579 , -0.6230169 , -0.05411236,  0.11218084,  0.20944953,
       -0.20439269,  0.13756047, -0.45810637,  0.06128663, -0.7068394 ,
        0.21369721,  0.14296761,  0.2475239 , -0.2520704 , -0.12699595,
       -0.04803002, -0.21700437, -0.19400993, -0.45585775, -0.02974468,
        0.4975682 , -0.03759894,  0.0382176 , -0.14872475, -0.20095265,
        0.28266242,  0.0845398 , -0.2956452 , -0.2371691 , -0.80624247,
        0.15711945, -0.28077704, -0.25714338, -0.04361391,  0.3300468 ,
       -0.11635113, -0.43941328, -0.07583012,  0.1754746 ,  0.32417488,
        0.09483029, -0.4148532 , -0.23878664, -0.07551412, -0.32269812,
        0.12700818,  0.25075105, -0.04256242, -0.34262484, -0.07183698,
        0.13333061,  0.00361201, -0.07537476,  0.11043669, -0.37876445,
        0.4320636 ,  0.15439911,  0.20283997, -0.4063872 ,  0.47

In [21]:
from tqdm import tqdm

In [22]:
x = []
for doc in tqdm(df['review'].values):
    x.append(document_vector(doc))

100%|██████████| 1000/1000 [00:08<00:00, 123.98it/s]


In [23]:
x = np.array(x)

In [24]:
x[0]

array([-0.2584384 ,  0.33628497,  0.15411253,  0.08223379,  0.03106987,
       -0.5955327 ,  0.21025276,  0.77334595, -0.24835569, -0.2774411 ,
       -0.2826579 , -0.6230169 , -0.05411236,  0.11218084,  0.20944953,
       -0.20439269,  0.13756047, -0.45810637,  0.06128663, -0.7068394 ,
        0.21369721,  0.14296761,  0.2475239 , -0.2520704 , -0.12699595,
       -0.04803002, -0.21700437, -0.19400993, -0.45585775, -0.02974468,
        0.4975682 , -0.03759894,  0.0382176 , -0.14872475, -0.20095265,
        0.28266242,  0.0845398 , -0.2956452 , -0.2371691 , -0.80624247,
        0.15711945, -0.28077704, -0.25714338, -0.04361391,  0.3300468 ,
       -0.11635113, -0.43941328, -0.07583012,  0.1754746 ,  0.32417488,
        0.09483029, -0.4148532 , -0.23878664, -0.07551412, -0.32269812,
        0.12700818,  0.25075105, -0.04256242, -0.34262484, -0.07183698,
        0.13333061,  0.00361201, -0.07537476,  0.11043669, -0.37876445,
        0.4320636 ,  0.15439911,  0.20283997, -0.4063872 ,  0.47

In [25]:
from sklearn.preprocessing import LabelEncoder

In [27]:
le = LabelEncoder()
y = le.fit_transform(df['sentiment'])

In [28]:
y

array([1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0,
       1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0,
       1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1,
       0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1,
       0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1,
       0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1,
       0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0,
       1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0,
       1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0,
       0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1,
       0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0,
       0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0,

In [29]:
from sklearn.model_selection import train_test_split

In [30]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size =0.2, random_state=1)

In [31]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [33]:
rf = RandomForestClassifier()
rf.fit(x_train,y_train)
y_pred = rf.predict(x_test)
accuracy_score(y_test,y_pred)


0.6